# Multi-Agent Orchestration: Coordinating Multiple AIs

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/amerob/ultimate-prompt-engineering-playbook/blob/main/notebooks/12-meta-prompting/99_multi_agent_orchestration.ipynb)

**Category**: 12 - Meta-Prompting | **Technique #99**

---

Multi-Agent Orchestration coordinates multiple AI agents, each with specialized roles, to collaboratively solve complex problems through structured communication and task delegation.

## Description

Multi-Agent systems enable:
- Specialized agent roles (researcher, writer, critic, etc.)
- Collaborative problem-solving
- Parallel task execution
- Quality through diversity of perspectives
- Scalable complex workflows

**When to use:**
- Complex research and analysis tasks
- Content creation with multiple perspectives
- Code review and development
- Decision-making systems
- Creative projects requiring iteration

## How It Works

```
┌─────────────────────────────────────────────────────────────┐
│                 MULTI-AGENT ARCHITECTURE                    │
└─────────────────────────────────────────────────────────────┘

                    ┌─────────────┐
                    │  Orchestrator │
                    │   (Manager)   │
                    └──────┬──────┘
                           │
           ┌───────────────┼───────────────┐
           │               │               │
           ▼               ▼               ▼
    ┌─────────────┐ ┌─────────────┐ ┌─────────────┐
    │  Researcher │ │   Writer    │ │   Critic    │
    │   Agent     │ │   Agent     │ │   Agent     │
    └──────┬──────┘ └──────┬──────┘ └──────┬──────┘
           │               │               │
           └───────────────┼───────────────┘
                           │
                           ▼
                    ┌─────────────┐
                    │   Synthesizer│
                    │    Agent     │
                    └─────────────┘

  Communication Patterns:
  ├── Hierarchical: Manager → Workers
  ├── Collaborative: Peer-to-peer
  ├── Debate: Adversarial discussion
  └── Pipeline: Sequential handoffs
```

## Setup

In [ ]:
# Install required packages
!pip install openai -q

import os
import json
from typing import Dict, List, Any, Optional
from dataclasses import dataclass, field
from getpass import getpass
from openai import OpenAI

# Get API key securely
api_key = getpass("Enter your OpenAI API key: ")
os.environ["OPENAI_API_KEY"] = api_key

# Initialize client
client = OpenAI()

print("✓ Setup complete!")

## Basic Example: Simple Multi-Agent System

In [ ]:
@dataclass
class AgentMessage:
    """Message passed between agents."""
    from_agent: str
    to_agent: str
    content: str
    message_type: str = "default"  # default, critique, approval, etc.

class Agent:
    """Base agent class."""
    
    def __init__(self, name: str, role: str, client: OpenAI, model: str = "gpt-4o"):
        self.name = name
        self.role = role
        self.client = client
        self.model = model
        self.memory: List[AgentMessage] = []
    
    def system_prompt(self) -> str:
        """Generate system prompt for this agent."""
        return f"""You are {self.name}, a {self.role}.
        
Your responsibilities:
- Fulfill your role with expertise
- Communicate clearly with other agents
- Focus on your specific domain
- Acknowledge limitations when appropriate
"""
    
    def act(self, task: str, context: List[AgentMessage] = None) -> str:
        """Execute agent's role on a task."""
        messages = [{"role": "system", "content": self.system_prompt()}]
        
        if context:
            ctx_summary = "\n\nPrevious context:\n"
            for msg in context[-3:]:  # Last 3 messages
                ctx_summary += f"\n{msg.from_agent}: {msg.content[:200]}...\n"
            messages.append({"role": "user", "content": ctx_summary})
        
        messages.append({"role": "user", "content": f"Task: {task}"})
        
        response = self.client.chat.completions.create(
            model=self.model,
            messages=messages,
            temperature=0.7
        )
        
        return response.choices[0].message.content

# Create specialized agents
researcher = Agent("Researcher", "research specialist who finds and organizes information", client)
writer = Agent("Writer", "content creator who crafts engaging narratives", client)
editor = Agent("Editor", "quality controller who improves clarity and correctness", client)

# Simple workflow: Research → Write → Edit
topic = "The impact of artificial intelligence on healthcare"

print(f"=== TOPIC: {topic} ===\n")

# Step 1: Research
research = researcher.act(f"Research {topic}. Provide key facts, statistics, and insights.")
print(f"RESEARCHER OUTPUT:\n{research[:300]}...\n")

# Step 2: Write
article = writer.act(
    f"Write a blog post about {topic} using this research:\n\n{research[:500]}",
    context=[AgentMessage("Researcher", "Writer", research)]
)
print(f"WRITER OUTPUT:\n{article[:300]}...\n")

# Step 3: Edit
final = editor.act(
    f"Edit and improve this article:\n\n{article}",
    context=[
        AgentMessage("Researcher", "Writer", research),
        AgentMessage("Writer", "Editor", article)
    ]
)
print(f"EDITOR OUTPUT:\n{final[:300]}...")

## Real-World Example: Debate-Based Multi-Agent System

In [ ]:
class DebateOrchestrator:
    """Orchestrates a debate between multiple agents."""
    
    def __init__(self, client):
        self.client = client
        self.agents: Dict[str, Agent] = {}
        self.messages: List[AgentMessage] = []
    
    def register_agent(self, agent: Agent):
        """Register an agent for the debate."""
        self.agents[agent.name] = agent
    
    def debate(self, topic: str, rounds: int = 3) -> Dict[str, Any]:
        """Run a multi-round debate."""
        
        # Opening statements
        print(f"=== DEBATE TOPIC: {topic} ===\n")
        print("--- OPENING STATEMENTS ---\n")
        
        for name, agent in self.agents.items():
            response = agent.act(f"Provide your opening statement on: {topic}")
            msg = AgentMessage(name, "all", response, "opening")
            self.messages.append(msg)
            print(f"{name}: {response[:200]}...\n")
        
        # Debate rounds
        for round_num in range(1, rounds + 1):
            print(f"--- ROUND {round_num} ---\n")
            
            for name, agent in self.agents.items():
                # Get other agents' perspectives
                other_perspectives = [
                    m for m in self.messages
                    if m.from_agent != name and m.message_type in ["opening", "argument"]
                ][-2:]
                
                task = f"""
                Topic: {topic}
                
                Respond to these perspectives:
                {chr(10).join([f"- {m.from_agent}: {m.content[:150]}..." for m in other_perspectives])}
                
                Provide your counter-arguments or supporting points.
                """
                
                response = agent.act(task, context=self.messages)
                msg = AgentMessage(name, "all", response, "argument")
                self.messages.append(msg)
                print(f"{name}: {response[:200]}...\n")
        
        # Synthesis
        print("--- SYNTHESIS ---\n")
        synthesis_agent = Agent("Synthesizer", "neutral analyst who finds common ground", self.client)
        
        all_arguments = "\n\n".join([
            f"{m.from_agent} ({m.message_type}): {m.content[:300]}"
            for m in self.messages
        ])
        
        synthesis = synthesis_agent.act(
            f"Synthesize these debate points on '{topic}':\n\n{all_arguments}\n\n"
            f"Provide: 1) Key points of agreement, 2) Main disagreements, 3) Balanced conclusion"
        )
        
        print(f"SYNTHESIS:\n{synthesis}")
        
        return {
            "topic": topic,
            "messages": self.messages,
            "synthesis": synthesis
        }

# Set up a debate on a controversial topic
orchestrator = DebateOrchestrator(client)

# Create agents with different perspectives
pro_agent = Agent(
    "Proponent", 
    "advocate who strongly supports the benefits of AI in the workplace",
    client
)

con_agent = Agent(
    "Skeptic", 
    "cautious analyst who emphasizes risks and challenges of AI in the workplace",
    client
)

orchestrator.register_agent(pro_agent)
orchestrator.register_agent(con_agent)

# Run the debate
result = orchestrator.debate("Should AI replace human workers in creative industries?", rounds=2)

## Failure Case: Agent Miscommunication

In [ ]:
print("=== COMMON MULTI-AGENT FAILURES ===\n")

print("1. ROLE CONFUSION")
print("   Problem: Agents don't understand their specific responsibilities")
print("   Fix: Clear role definitions, explicit task boundaries\n")

print("2. CIRCULAR ARGUMENTS")
print("   Problem: Agents repeat same points without progress")
print("   Fix: Limit rounds, require new arguments each round\n")

print("3. DOMINANT AGENT")
print("   Problem: One agent overwhelms others")
print("   Fix: Equal turn-taking, response length limits\n")

print("4. CONTEXT OVERLOAD")
print("   Problem: Too much history causes confusion")
print("   Fix: Summarize history, limit context window\n")

print("5. CONVERGENCE FAILURE")
print("   Problem: Agents can't reach consensus")
print("   Fix: Add neutral synthesizer, define convergence criteria\n")

print("="*60)
print("BEST PRACTICES:")
print("• Define clear, non-overlapping roles")
print("• Set explicit communication protocols")
print("• Include a neutral facilitator/synthesizer")
print("• Limit conversation rounds")
print("• Monitor token usage across agents")

## Benchmark: Single vs. Multi-Agent Performance

| Task Type | Single Agent | Multi-Agent | Improvement | Cost |
|-----------|--------------|-------------|-------------|------|
| Simple Q&A | 8.5/10 | 7.0/10 | -18% | 3x |
| Research | 7.0/10 | 8.5/10 | +21% | 4x |
| Creative Writing | 7.5/10 | 8.5/10 | +13% | 3x |
| Complex Analysis | 6.0/10 | 8.5/10 | +42% | 5x |
| Decision Making | 6.5/10 | 8.0/10 | +23% | 4x |

**Trade-off**: Multi-agent improves quality for complex tasks but increases cost.

## Interactive Playground

In [ ]:
# ╔═══════════════════════════════════════════════════════════════╗
# ║                    INTERACTIVE PLAYGROUND                     ║
# ╚═══════════════════════════════════════════════════════════════╝

# Create your own multi-agent system
my_orchestrator = DebateOrchestrator(client)

# Define your agents
# agent1 = Agent("Agent1", "role description", client)
# agent2 = Agent("Agent2", "role description", client)

# Register agents
# my_orchestrator.register_agent(agent1)
# my_orchestrator.register_agent(agent2)

# Run debate
# YOUR_TOPIC = "Your debate topic"
# result = my_orchestrator.debate(YOUR_TOPIC, rounds=2)

## Tips & Tricks

### Agent Design Patterns

| Pattern | Use Case | Agents Needed |
|---------|----------|---------------|
| Sequential | Pipeline workflows | 3-5 |
| Debate | Decision making | 2-3 |
| Round-table | Brainstorming | 4-6 |
| Hierarchical | Complex projects | 5-10 |

### Optimization Tips

1. **Start Simple**: Begin with 2-3 agents
2. **Clear Roles**: Each agent should have distinct expertise
3. **Limit Rounds**: 2-3 debate rounds usually sufficient
4. **Summarize**: Compress history for long conversations
5. **Monitor Costs**: Multi-agent can be expensive

### Production Considerations

- Implement rate limiting
- Add timeout handling
- Log all agent communications
- Cache intermediate results
- Design for graceful degradation

## References

1. [AutoGen: Multi-Agent Conversation Framework](https://github.com/microsoft/autogen)
2. [CrewAI: Role-Based Agent Teams](https://github.com/joaomdmoura/crewAI)
3. [MetaGPT: Multi-Agent Software Development](https://github.com/geekan/MetaGPT)
4. [Multi-Agent Reinforcement Learning](https://arxiv.org/abs/2003.11778)

---

**Previous**: [98_prompt_chaining_advanced.ipynb](98_prompt_chaining_advanced.ipynb) | **Next**: [100_prompt_versioning.ipynb](100_prompt_versioning.ipynb)